# Analysis 3 — Step 0: channel_title 복구 & 머지

- **목적**: 정제 때 드롭된 `channel_title`을 원본에서 복구해 정본에 머지 (누수 아님 — 업로드시점 정보).
- **입력**: `../dataset/cleaned_USvideos.csv`(정본) + `../dataset/USvideos.csv`(복구원)
- **산출**: `step0_channel_merge.csv`
- 계획: `docs/Analysis3_channel/phase1/plan.md` · 배경: `Background.md`

In [1]:
import pandas as pd
import numpy as np

DEDUP = lambda d: d.sort_values("views").drop_duplicates("video_id", keep="last")

In [2]:
# 로드
clean = pd.read_csv("../dataset/cleaned_USvideos.csv")
orig = pd.read_csv("../dataset/USvideos.csv")
print("cleaned:", clean.shape, "| orig:", orig.shape)

cleaned: (6249, 26) | orig: (40949, 16)


In [3]:
# dedup → channel_title 머지 (video_id 기준 left join)
clean = DEDUP(clean).reset_index(drop=True)
orig_map = (DEDUP(orig)[["video_id", "channel_title"]]
            .dropna(subset=["channel_title"]).reset_index(drop=True))
merged = clean.merge(orig_map, on="video_id", how="left")

n_total = len(merged)
n_matched = merged["channel_title"].notna().sum()
print(f"매칭률: {n_matched}/{n_total} = {n_matched / n_total:.4%}")
print(f"미매칭(channel 결측): {n_total - n_matched}")
merged["channel_title"] = merged["channel_title"].fillna("__UNKNOWN__")

매칭률: 6249/6249 = 100.0000%
미매칭(channel 결측): 0


In [4]:
# 채널 분포 — high-cardinality / 단일 등장 비율 점검
vc = merged["channel_title"].value_counts()
n_channels = merged.loc[merged["channel_title"] != "__UNKNOWN__", "channel_title"].nunique()
n_single = (vc[vc.index != "__UNKNOWN__"] == 1).sum()
print(f"고유 채널 수: {n_channels}")
print(f"단일 등장 채널: {n_single} ({n_single / n_channels:.2%} of channels)")
print("채널당 영상 수 분포:")
print(vc.describe().round(2).to_string())
print("\n상위 등장 채널 Top10:")
print(vc.head(10).to_string())

고유 채널 수: 2100
단일 등장 채널: 1294 (61.62% of channels)
채널당 영상 수 분포:
count    2100.00
mean        2.98
std         5.85
min         1.00
25%         1.00
50%         1.00
75%         2.00
max        84.00

상위 등장 채널 Top10:
channel_title
ESPN                                      84
TheEllenShow                              74
The Tonight Show Starring Jimmy Fallon    72
Jimmy Kimmel Live                         70
Netflix                                   58
The Late Show with Stephen Colbert        58
NBA                                       55
CNN                                       52
Vox                                       47
The Late Late Show with James Corden      46


In [5]:
# 저장
merged.to_csv("step0_channel_merge.csv", index=False)
print("[저장] step0_channel_merge.csv", merged.shape)

[저장] step0_channel_merge.csv (6249, 27)


## 점검 메모

- 매칭률 **100%** (6249/6249, 미매칭 0) → `__UNKNOWN__` fallback 미발동.
- 고유 채널 **2,100**, 단일 등장 **61.6%** → high-cardinality(Background §5-4 예견) → Step 1 OOF에서 카테고리 평균 fallback 필요.